# 傅里叶变换与频谱表示

学习目标：定位合成信号中的频率分量，正确解释频率轴、幅值与功率，并检查逆变换和采样边界。

前置知识：复数、正弦信号、采样率、频率与周期、数组索引和数值容差。

运行环境：Python 3.12、NumPy 2.5。

环境准备：[环境配置与运行](README.md)。

工作目录：本 Notebook 所在目录；重启内核后从上到下运行。

示例使用等间隔采样的合成信号，后续单元沿用首次导入的 np。除二维选学示例外，时间单位为秒、频率单位为 Hz、信号单位为 V。

## 1 找出周期信号的频率

设采样率 fs=8 Hz，即每秒取 8 个样本，采样间隔 dt=1/fs=0.125 秒。取 n=8 个点，时刻为 0、dt、…、(n−1)dt。下面的余弦信号频率为 2 Hz、幅值为 1 V，每隔 4 个样本重复一次。

np.fft.fft 把离散时间信号变成复数频谱。FFT 是快速计算离散傅里叶变换（DFT）的算法；np.fft.fftfreq(n, d=dt) 给出各输出位置对应的频率。先在正频率范围内找模最大的项，检查是否定位到已知频率。

In [1]:
import numpy as np

fs = 8.0
dt = 1 / fs
n = 8
t = np.arange(n) * dt
signal = np.cos(2 * np.pi * 2 * t)
spectrum = np.fft.fft(signal)
frequency = np.fft.fftfreq(n, d=dt)
positive = frequency > 0
peak = np.argmax(np.abs(spectrum[positive]))

print(np.round(signal, 6))  # 约 [1, 0, -1, 0, 1, 0, -1, 0]。
print(frequency[positive][peak])  # 2.0 Hz，与设定频率一致。
print(np.round(np.abs(spectrum), 6))  # +2 Hz、-2 Hz 两处模均为 4。

[ 1.  0. -1. -0.  1.  0. -1. -0.]
2.0
[0. 0. 4. 0. 0. 0. 4. 0.]


## 2 输出顺序与逆变换

默认 fft 的零频率项是样本之和，随后放正频率，再放负频率；输出本身并不按频率从小到大排列。偶数长度时，中间项代表奈奎斯特频率（Nyquist frequency）fs/2，fftfreq 将它标成 −fs/2。

对于 n 个原始样本，频率网格间隔 Δf=fs/n。这里用 n×dt=1 秒表示 DFT 的记录时长，最后一个采样时刻仍是 0.875 秒。若省略 d，频率单位是“周期／样本”，不能直接当成 Hz。

继续查看上一例的频率标签。实数信号具有共轭对称性：负频率系数是对应正频率系数的复共轭；负频率不是另一条独立的余弦信号。

In [2]:
print(frequency)  # [0. 1. 2. 3. -4. -3. -2. -1.] Hz
print(np.fft.fftfreq(n))  # [0, 0.125, 0.25, 0.375, -0.5, ...] 周期／样本。
print(fs / n, n * dt, t[-1])  # 1.0 Hz、1.0 秒、0.875 秒。
print(np.allclose(spectrum[1:4], np.conj(spectrum[-1:-4:-1]),
                  rtol=0, atol=1e-12))  # True：成对的正负频率共轭对称。

[ 0.  1.  2.  3. -4. -3. -2. -1.]
[ 0.     0.125  0.25   0.375 -0.5   -0.375 -0.25  -0.125]
1.0 1.0 0.875
True


np.fft.ifft 从完整复数频谱还原信号。逆变换需要保留复数系数，不能只把频谱的模传回去，因为取模会丢掉相位信息。

下面先检查复数结果与原信号的误差，再观察实部。对本章个位数幅值、至多几十点的 float64 合成输入，使用 1e-12 作为重构绝对容差，rtol=0；它为浮点舍入留出余量，不是实际仪器的测量精度。

In [3]:
restored = np.fft.ifft(spectrum)

print(restored.shape, restored.dtype)  # (8,) complex128
print(np.max(np.abs(restored - signal)))  # 接近零。
print(np.max(np.abs(restored.imag)))  # 接近零；先检查，再只看实部。
print(np.allclose(restored, signal, rtol=0, atol=1e-12))  # True
print(np.round(restored.real, 6))  # 还原原来的余弦样本。

(8,) complex128
4.930380657631324e-32
0.0
True
[ 1.  0. -1. -0.  1.  0. -1. -0.]


## 3 实数变换与奇偶长度

对实数输入，rfft 只保留非负频率部分，输出长度为 n//2+1；频率轴由 rfftfreq 生成。若 n 为偶数，最后一项对应 fs/2，是实数；rfftfreq 将这项标为正频率。

下面使用同一个 8 点信号，irfft 的 n 明确指定原始样本数。普通 fft 适用于一般复数输入；rfft 用于实数输入。

In [4]:
signal = np.array([1.0, 0.0, -1.0, 0.0, 1.0, 0.0, -1.0, 0.0])
dt = 0.125
half_spectrum = np.fft.rfft(signal)
frequency = np.fft.rfftfreq(signal.size, d=dt)
restored = np.fft.irfft(half_spectrum, n=signal.size)

print(half_spectrum.shape, frequency)  # (5,) [0. 1. 2. 3. 4.]
print(half_spectrum)  # 2 Hz 项为 4，其他项为 0。
print(restored.shape, restored.dtype)  # (8,) float64
print(np.allclose(restored, signal, rtol=0, atol=1e-12))  # True

(5,) [0. 1. 2. 3. 4.]
[0.+0.j 0.+0.j 4.-0.j 0.+0.j 0.+0.j]
(8,) float64
True


若 n 为奇数，rfft 没有恰好位于 fs/2 的项，最后一项通常是复数。仅从半谱长度无法判断原输入是奇数还是偶数长度。

irfft 不给 n 时默认输出 2×(m−1) 点，其中 m 是半谱长度，相当于假设原长度为偶数。下面 7 点输入的半谱有 4 项；省略 n 会得到 6 点。保存频谱时应同时保留原始长度和采样间隔。

In [5]:
n = 7
fs = 7.0
t = np.arange(n) / fs
signal = np.sin(2 * np.pi * 3 * t)
half_spectrum = np.fft.rfft(signal)
wrong_length = np.fft.irfft(half_spectrum)
restored = np.fft.irfft(half_spectrum, n=n)

print(np.fft.rfftfreq(n, d=1 / fs))  # [0. 1. 2. 3.]，未到奈奎斯特频率 3.5 Hz。
print(np.round(half_spectrum[-1], 6))  # 约 -3.5j，最后一项不是纯实数。
print(wrong_length.shape, restored.shape)  # (6,) (7,)
print(np.allclose(restored, signal, rtol=0, atol=1e-12))  # True

[0. 1. 2. 3.]
(-0-3.5j)
(6,) (7,)
True


## 4 归一化约定

norm 控制正、逆变换中的缩放。表中的 n 是变换长度；一对正逆变换使用相同 norm，才按对应约定还原。

| norm | 中文含义 | 正变换缩放 | 逆变换缩放 |
| --- | --- | --- | --- |
| backward | 默认：逆变换归一化 | 1 | 1/n |
| forward | 正变换归一化 | 1/n | 1 |
| ortho | 正交归一化 | 1/√n | 1/√n |

下面 4 点信号的默认频谱在第 1、3 项均为 2。改变 norm 会改变系数大小，但匹配的逆变换仍能还原原信号。

In [6]:
signal = np.array([1.0, 0.0, -1.0, 0.0])
for mode in ("backward", "forward", "ortho"):
    spectrum = np.fft.fft(signal, norm=mode)
    restored = np.fft.ifft(spectrum, norm=mode)
    print(mode, np.round(np.abs(spectrum), 6),
          np.allclose(restored, signal, rtol=0, atol=1e-12))
# 三种模式的非零模依次为 2、0.5、1；重构检查均为 True。

backward [0. 2. 0. 2.] True
forward [0.  0.5 0.  0.5] True
ortho [0. 1. 0. 1.] True


## 5 幅值与功率

默认 fft 的模没有除以点数，不能直接当作原余弦的幅值。对未加窗、频率恰好落在网格上的实数正弦或余弦，先把 rfft 的模除以 n，再把成对正负频率对应的内部项乘 2，可得到单边幅值。

零频率项（直流）不乘 2；偶数长度的奈奎斯特项也不乘 2。奇数长度没有奈奎斯特项，因此除了直流，其余保留项都乘 2。

下面是直流 1 V、2 Hz 余弦幅值 2 V，以及奈奎斯特频率 4 Hz 余弦幅值 0.5 V 的叠加。此处 4 Hz 项是特定相位的离散样本，不能据它恢复任意相位的连续信号。

In [7]:
fs = 8.0
n = 8
t = np.arange(n) / fs
signal = 1 + 2 * np.cos(2 * np.pi * 2 * t) + 0.5 * np.cos(2 * np.pi * 4 * t)
half_spectrum = np.fft.rfft(signal)
amplitude = np.abs(half_spectrum) / n
# 单边幅值补回另一半频谱；直流项不加倍，偶数长度还保留奈奎斯特项。
if n % 2 == 0:
    amplitude[1:-1] *= 2
else:
    amplitude[1:] *= 2

print(np.fft.rfftfreq(n, d=1 / fs))  # [0. 1. 2. 3. 4.] Hz
print(np.round(amplitude, 6))  # [1. 0. 2. 0. 0.5] V

[0. 1. 2. 3. 4.]
[1.  0.  2.  0.  0.5]


功率类量使用模的平方，单位和幅值不同。本章以样本均方值 mean(signal²) 表示平均功率，单位为 V²；没有给定负载电阻时，不把它标成瓦特。

默认、未加窗 DFT 的每个双边频率项对均方值的贡献为 abs(spectrum)²/n²。压成单边表示时，先平方再合并正负频率的功率，也只把内部项乘 2。它不同于“先将幅值乘 2，再平方”，也不是单位为 V²/Hz 的功率谱密度。

继续使用上一例，检查频率贡献之和与直接计算的均方值一致。

In [8]:
power = np.abs(half_spectrum) ** 2 / n ** 2
power[1:-1] *= 2  # 本例 n=8，直流与奈奎斯特项不加倍。

print(np.round(power, 6))  # [1. 0. 2. 0. 0.25] V²
print(np.sum(power), np.mean(signal ** 2))  # 两者均约 3.25 V²。
print(np.allclose(np.sum(power), np.mean(signal ** 2), rtol=0, atol=1e-12))
print(amplitude[2] ** 2, power[2])  # 4 与 2：幅值平方不等于该余弦的平均功率。

[1.   0.   2.   0.   0.25]
3.25 3.25
True
4.0 2.0


奇数长度时，最高正频率项仍有对应的负频率项，幅值缩放也要乘 2。下面把样本数改为 7，频率 3 Hz 仍恰好落在网格上，用它核对最后一项的处理。

In [9]:
n = 7
fs = 7.0
t = np.arange(n) / fs
signal = np.sin(2 * np.pi * 3 * t)
amplitude = np.abs(np.fft.rfft(signal)) / n
amplitude[1:] *= 2

print(np.fft.rfftfreq(n, d=1 / fs))  # [0. 1. 2. 3.] Hz
print(np.round(amplitude, 6))  # [0. 0. 0. 1.] V，最后一项恢复幅值 1。

[0. 1. 2. 3.]
[0. 0. 0. 1.]


## 6 移动频谱的显示顺序

fftshift 把零频率移到中间，常用于按负频率到正频率观察完整频谱。频率轴与系数必须一起移动；这个操作只重排位置，不改变信号的频率内容。

逆变换前用 ifftshift 恢复标准顺序。对于奇数长度，fftshift 和 ifftshift 的移动量不同，连续调用两次 fftshift 不能代替恢复操作。下面特意使用 5 点输入观察这一边界。

In [10]:
signal = np.array([0.0, 1.0, 0.0, -1.0, 0.0])
spectrum = np.fft.fft(signal)
frequency = np.fft.fftfreq(signal.size, d=0.2)
centered = np.fft.fftshift(spectrum)

print(np.fft.fftshift(frequency))  # [-2. -1. 0. 1. 2.] Hz
restored = np.fft.ifft(np.fft.ifftshift(centered))
print(np.allclose(restored, signal, rtol=0, atol=1e-12))  # True
labels = np.arange(5)
print(np.fft.fftshift(np.fft.fftshift(labels)))  # [1 2 3 4 0]，没有恢复。
print(np.fft.ifftshift(np.fft.fftshift(labels)))  # [0 1 2 3 4]

[-2. -1.  0.  1.  2.]
True
[1 2 3 4 0]
[0 1 2 3 4]


## 7 采样限制与混叠

奈奎斯特频率是 fs/2。等间隔采样后，超过可区分频带的连续信号可能与较低频率产生相同样本，称为混叠（aliasing）。FFT 只能分析已有样本，不能凭空识别混叠前的频率。

实际采样要限制输入带宽；本章合成信号的常规分析要求最高频率严格低于 fs/2，并为实际采样留出余量。恰好在 fs/2 处，相位也有特殊影响：正弦可能处处采到零，余弦却交替为正负峰值。

下图把连续曲线与实际采样点分开：实线和虚线在采样点之间不同，8 个深色采样点却完全重合。这里只给 FFT 这些点，就无法区分 2 Hz 与 6 Hz。

![2 Hz 与 6 Hz 余弦在 8 Hz 采样下具有共同采样点的混叠示意图](image/25-aliasing.png)

In [11]:
fs = 8.0
t = np.arange(8) / fs
low = np.cos(2 * np.pi * 2 * t)
high = np.cos(2 * np.pi * 6 * t)
nyquist_sine = np.sin(2 * np.pi * (fs / 2) * t)
nyquist_cosine = np.cos(2 * np.pi * (fs / 2) * t)

print(np.allclose(low, high, rtol=0, atol=1e-12))  # True：2 Hz 与 6 Hz 样本无法区分。
print(np.round(nyquist_sine, 6))  # 约全零，并不代表连续正弦不存在。
print(np.round(nyquist_cosine, 6))  # [1. -1. 1. -1. 1. -1. 1. -1.]

True
[ 0.  0. -0.  0. -0.  0. -0.  0.]
[ 1. -1.  1. -1.  1. -1.  1. -1.]


## 8 有限观测与补零

有限时长截取信号会使频谱向邻近频率扩散，称为频谱泄漏（spectral leakage）。若一个正弦分量恰好落在 DFT 网格上，未加窗时可以集中在对应网格项；没有落在网格上时，多个频率项可能都非零，不能把每项都解释成一个真实信号源。

下面以 8 Hz 采样 8 点，网格间隔为 1 Hz。对比 2 Hz 与 1.5 Hz 的余弦，后者不落在这个网格上。

In [12]:
fs = 8.0
n = 8
t = np.arange(n) / fs
on_grid = np.cos(2 * np.pi * 2 * t)
off_grid = np.cos(2 * np.pi * 1.5 * t)

print(np.round(np.abs(np.fft.rfft(on_grid)), 3))  # [0. 0. 4. 0. 0.]
print(np.round(np.abs(np.fft.rfft(off_grid)), 3))  # 多个网格项非零。

[0. 0. 4. 0. 0.]
[1.    2.398 2.798 1.192 1.   ]


fft 或 rfft 的 n 参数大于输入长度时，在末尾补零（zero padding）；小于输入长度时会截去尾部。这里继续使用 1.5 Hz 的 8 个观测样本，将变换长度增加到 32，频率网格从 1 Hz 变为 0.25 Hz。

补零只让同一段观测的频谱取样更密，没有增加真实观测时间，也不会提高分辨相近频率分量的物理能力。下面原网格上的复系数保持一致；新出现的网格点来自同一段数据。

下图左侧对比落在与偏离网格的余弦，右侧对比同一组 1.5 Hz 样本补零前后的频谱。纵轴统一除以原样本数 8，正频率项未乘 2，不能直接读成单边余弦幅值。右图空心圆与对应橙色点重合；连线只帮助观察趋势，不表示新增了观测数据。

![原网格上的频谱泄漏与补零后的更密频谱取样对照图](image/25-leakage-zero-padding.png)

In [13]:
original = np.fft.rfft(off_grid)
padded = np.fft.rfft(off_grid, n=32)
original_frequency = np.fft.rfftfreq(8, d=1 / fs)
padded_frequency = np.fft.rfftfreq(32, d=1 / fs)

print(original_frequency[1], padded_frequency[1])  # 1.0、0.25 Hz。
print(original_frequency[np.argmax(np.abs(original))])  # 2.0 Hz。
print(padded_frequency[np.argmax(np.abs(padded))])  # 本例为 1.5 Hz，峰值位置取样更细。
print(np.allclose(padded[::4], original, rtol=0, atol=1e-12))  # True
short = np.fft.irfft(np.fft.rfft(off_grid, n=4), n=4)
print(np.allclose(short, off_grid[:4], rtol=0, atol=1e-12))  # True，只保留前 4 点。

1.0 0.25
2.0
1.5
True
True


## 9 选学：窗函数

窗函数是在变换前乘到信号上的权重。np.hanning(n) 生成两端逐渐降到零的 Hann 窗。它可以压低远离主峰的旁瓣，但代价是主瓣变宽，不能同时保证抑制泄漏和分开任意相近的频率。

加窗也会改变幅值。下面为比较幅值尺度，未加窗时除以样本数，加窗时除以窗权重之和。这里只比较正频率的双边幅值贡献，没有乘 2；这类幅值修正不等于功率归一化，也不保证偏离网格的峰值恰好等于输入幅值。

使用 32 Hz 采样的 32 点、5.5 Hz 信号，观察远处 10 Hz 及以上的幅值。这个范围是本例选定的旁瓣观察区，不是通用分界。

In [14]:
fs = 32.0
n = 32
t = np.arange(n) / fs
signal = np.cos(2 * np.pi * 5.5 * t)
window = np.hanning(n)
frequency = np.fft.rfftfreq(n, d=1 / fs)
plain = np.abs(np.fft.rfft(signal)) / n
# 加窗后的幅值用窗权重之和缩放，便于与未加窗结果比较。
windowed = np.abs(np.fft.rfft(signal * window)) / window.sum()
far = frequency >= 10

print(window[[0, -1]])  # [0. 0.]，两端权重为零。
print(np.max(plain[far]), np.max(windowed[far]))  # 约 0.0460 与 0.00183，远处旁瓣降低。
print(np.round(plain[3:9], 4))  # 观察 3—8 Hz 的主峰附近。
print(np.round(windowed[3:9], 4))  # 主峰附近的分布也改变，不能只看旁瓣改善。
restored = np.fft.irfft(np.fft.rfft(signal * window), n=n)
print(np.allclose(restored, signal * window, rtol=0, atol=1e-12))  # True：还原的是加窗信号。

[0. 0.]
0.046035483492540046 0.0018285882850678304
[0.0575 0.0988 0.3103 0.3269 0.1152 0.0733]
[0.013  0.0974 0.4288 0.4288 0.0975 0.0131]
True


## 10 选学：二维变换

fft2、ifft2 默认沿最后两个轴变换；fftn、ifftn 可推广到更多指定轴。若一个轴表示批次，不能把它误当成需要变换的空间轴。

下面是 3 行、4 列的规则网格，行表示竖直位置，列表示水平位置，相邻位置都间隔 1 米。每行都按 [1, −1, 1, −1] 变化，竖直方向不变，所以二维频谱只在“竖直 0 周期／米、水平奈奎斯特频率”处非零。频率单位由空间间隔决定，此处不是 Hz。

In [15]:
grid = np.array([[1.0, -1.0, 1.0, -1.0],
                 [1.0, -1.0, 1.0, -1.0],
                 [1.0, -1.0, 1.0, -1.0]])
spectrum = np.fft.fft2(grid, axes=(0, 1))
restored = np.fft.ifft2(spectrum, axes=(0, 1))

print(spectrum.shape)  # (3, 4)
print(np.round(np.abs(spectrum), 6))  # 仅 [0, 2] 为 12。
print(np.fft.fftfreq(3, d=1.0))  # 竖直频率 [0, 1/3, -1/3] 周期／米。
print(np.fft.fftfreq(4, d=1.0))  # 水平频率 [0, 0.25, -0.5, -0.25] 周期／米。
print(np.allclose(restored, grid, rtol=0, atol=1e-12))  # True

(3, 4)
[[ 0.  0. 12.  0.]
 [ 0.  0.  0.  0.]
 [ 0.  0.  0.  0.]]
[ 0.          0.33333333 -0.33333333]
[ 0.    0.25 -0.5  -0.25]
True


## 本章小结

（1）频谱必须与样本数、采样间隔和归一化约定一起解释。fftfreq 默认不是 Hz，完整 FFT 输出也不是按频率递增排列。

（2）实数输入可用 rfft；irfft 显式保留原长度，尤其不要让奇数长度被默认的偶数长度替代。

（3）幅值、幅值平方和平均功率的缩放不同。直流和偶数长度的奈奎斯特项不按内部正频率项加倍。

（4）fftshift 用于重排显示，逆变换前用 ifftshift 恢复。保留复系数才能保留相位信息。

（5）混叠源于采样限制；补零增加频率网格点，不增加真实观测。窗函数在旁瓣、主瓣宽度和幅值尺度之间引入取舍。

## 练习

（1）以 12 Hz 采样 12 点，构造幅值 3 V、频率 2 Hz 的余弦。用 rfft 定位峰值并恢复单边幅值，再用 irfft 还原。分别检查频率、幅值、形状和重构误差。

In [16]:
fs = 12.0
n = 12
t = np.arange(n) / fs
signal = 3 * np.cos(2 * np.pi * 2 * t)

# 在此计算频率轴、单边幅值和逆变换。
# 检查峰值为 2 Hz、幅值约 3 V，重构形状为 (12,)，最大误差不超过 1e-12。

（2）先预测下面两次逆变换的输出长度，再运行核对。说明为何只有半谱长度还不够，以及最后一个半谱系数能否被当作纯实数的奈奎斯特项。

In [17]:
n = 9
signal = np.sin(2 * np.pi * 4 * np.arange(n) / n)
spectrum = np.fft.rfft(signal)

print(np.fft.irfft(spectrum).shape)
print(np.fft.irfft(spectrum, n=n).shape)
print(spectrum[-1])
# 在此检查指定 n 后的重构误差，并解释预测与观察。

(8,)
(9,)
(-2.8815264254582844e-15-4.499999999999998j)


（3）已有 16 个样本，想分辨 4 Hz 与 4.25 Hz 两个相近分量。可选方案是把现有数据补零到 128 点，或按原采样率实际采集更长时间。哪一种增加了观测信息？说明选择理由，再比较补零前后原网格上的系数；不要仅凭网格间隔变小就声称两分量已被分开。

In [18]:
fs = 16.0
t = np.arange(16) / fs
signal = np.cos(2 * np.pi * 4 * t) + np.cos(2 * np.pi * 4.25 * t)

# 在此比较 n=16 与 n=128 的 rfft；每隔 8 个补零后网格点核对原系数。
# 写下采集方案的选择理由，并区分网格间隔与分辨相近频率的能力。

（4）下面实数信号恰好落在网格上。求出 0 Hz 和 3 Hz 的单边幅值及平均功率贡献，检查贡献之和等于直接计算的均方值。解释为什么 3 Hz 处不能直接把单边幅值平方当作功率贡献。

In [19]:
fs = 12.0
n = 12
t = np.arange(n) / fs
signal = 2 + 4 * np.cos(2 * np.pi * 3 * t)

# 在此分别计算幅值和功率；直流与奈奎斯特位置不加倍。
# 用 rtol=0、atol=1e-12 核对功率贡献总和与 mean(signal ** 2)。

### 重点练习提示

对应第（2）题。先独立完成，再按需要查看提示。

（1）从半谱的 5 个系数分别反推可能的奇数和偶数原长度。

（2）irfft 不传 n 时假定偶数长度，最后一个系数的位置也随这个假定改变。

### 重点练习参考解析

对应第（2）题。

长度 9 的实信号产生长度 5 的半谱；长度 8 的实信号也会产生长度 5 的半谱，所以仅凭半谱长度不能区分两者。默认 irfft 返回 2×(5−1)=8 个点；显式传 n=9 才返回 (9,) 并在本题绝对容差 1e-12 内重构原信号。

本题最后一个系数对应频率索引 4，数值接近 −4.5j，不是纯实数的奈奎斯特项。对奇数长度没有恰好位于奈奎斯特频率的这一项；默认偶数解释会把末项按纯实数处理，因此不能只在逆变换后补一个点来恢复信息。

## 参考与引用来源

| 网站 | 本章参考内容与定位 |
| --- | --- |
| NumPy 官方文档（numpy.org） | NumPy 2.5，核查日期：2026-09-20。[Discrete Fourier Transform](https://numpy.org/doc/2.5/reference/routines.fft.html) 的 Background information、Implementation details、Normalization、Real and Hermitian transforms、Higher dimensions：DFT、输出顺序、幅度与相位、归一化和共轭对称；[fft](https://numpy.org/doc/2.5/reference/generated/numpy.fft.fft.html)、[ifft](https://numpy.org/doc/2.5/reference/generated/numpy.fft.ifft.html) 的 Parameters、Notes：变换长度、截断、补零和逆变换；[rfft](https://numpy.org/doc/2.5/reference/generated/numpy.fft.rfft.html)、[irfft](https://numpy.org/doc/2.5/reference/generated/numpy.fft.irfft.html) 的 Returns、Notes：实输入、奇偶长度与逆变换 n；[fftfreq](https://numpy.org/doc/2.5/reference/generated/numpy.fft.fftfreq.html)、[rfftfreq](https://numpy.org/doc/2.5/reference/generated/numpy.fft.rfftfreq.html)：频率单位与排列；[fftshift](https://numpy.org/doc/2.5/reference/generated/numpy.fft.fftshift.html)、[ifftshift](https://numpy.org/doc/2.5/reference/generated/numpy.fft.ifftshift.html)：显示重排与奇数长度；[hanning](https://numpy.org/doc/2.5/reference/generated/numpy.hanning.html) 的 Notes：Hann 窗与端点；[fft2](https://numpy.org/doc/2.5/reference/generated/numpy.fft.fft2.html)、[ifft2](https://numpy.org/doc/2.5/reference/generated/numpy.fft.ifft2.html) 的 Parameters、Notes：变换轴与二维往返；[allclose](https://numpy.org/doc/2.5/reference/generated/numpy.allclose.html) 的 Notes：数值检查的绝对与相对容差。 |
| SciPy 官方文档（docs.scipy.org） | SciPy 1.18 [Signal Processing：Spectral Analysis](https://docs.scipy.org/doc/scipy/tutorial/signal.html#spectral-analysis)，Continuous Signal、Sampled Sine Signal 小节：采样间隔、奈奎斯特频率、混叠、频谱泄漏、幅值／功率及单位（第 7、8 节原创图按本章相同输入计算，相关来源于 2026-09-21 复核）；式 (6)、(7) 及其后窗口归一化、矩形窗功率公式：默认 DFT 的均方值贡献、窗函数的旁瓣与主瓣取舍。本章代码仅使用 NumPy，不需要安装 SciPy。 |
| MathWorks 官方文档（mathworks.com） | [Amplitude Estimation and Zero Padding](https://www.mathworks.com/help/signal/ug/amplitude-estimation-and-zero-padding.html)：单边幅值缩放中直流与奈奎斯特项的例外，以及补零细化频谱取样但不提高频谱分辨率的条件说明。 |